# Lab 4 — Time-Aware RAG
## Decay Exponencial Temporal no OpenSearch

**Aula 11 · MBA RAG & CAG Aplicados a Direito e Segurança Pública**

**Objetivo:** Implementar um sistema RAG com consciência temporal para o domínio jurídico. Documentos mais recentes recebem scores mais altos via função de decay exponencial, evitando a recuperação de normas revogadas ou desatualizadas.

**Duração estimada:** 45 minutos

---

### O Problema no Domínio Jurídico

```
Sem Time-Aware RAG:
Query: "Pena para homicídio qualificado"
  → Retorna texto do CP de 1940 (original) ← ERRADO
  → Retorna texto após Pacote Anticrime 2019 ← CORRETO

Com Time-Aware RAG (decay exponencial):
Query: "Pena para homicídio qualificado"
  → Score 2019: 0.95 × decay(0d) = 0.95 × 1.0 = 0.95 ← PRIMEIRO
  → Score 1940: 0.95 × decay(84y) = 0.95 × 0.001 = 0.001 ← ÚLTIMO
```

## Etapa 0 — Instalação

In [ ]:
!pip install -q opensearch-py==2.4.2
!pip install -q sentence-transformers==3.0.0
!pip install -q python-dateutil==2.9.0

print("✅ Dependências instaladas!")

## Etapa 1 — Corpus Jurídico com Datas

In [ ]:
from datetime import datetime, timedelta

# Corpus com documentos de diferentes épocas
# Simula um índice real com legislação histórica e atual

documentos_temporais = [
    # --- Homicídio Qualificado ---
    {
        "id": "CP_1940_homicidio",
        "titulo": "Código Penal 1940 - Homicídio Qualificado",
        "texto": """Art. 121, §2º: Se o homicídio é cometido mediante paga ou promessa de recompensa, 
        ou por outro motivo torpe; por motivo fútil; com emprego de veneno, fogo, explosivo, asfixia, 
        tortura ou outro meio insidioso ou cruel, ou de que possa resultar perigo comum; à traição, 
        de emboscada, ou mediante dissimulação ou outro recurso que dificulte ou torne impossível a 
        defesa do ofendido; para assegurar a execução, a ocultação, a impunidade ou vantagem de outro 
        crime: Pena - reclusão, de doze a trinta anos.""",
        "data_publicacao": "1940-12-07",
        "tipo": "legislacao",
        "vigente": False,
        "revogado_por": "Lei 13.964/2019"
    },
    {
        "id": "CP_2019_homicidio",
        "titulo": "Código Penal - Pacote Anticrime 2019 - Homicídio",
        "texto": """Art. 121, §2º (redação dada pelo Pacote Anticrime - Lei 13.964/2019): 
        Se o homicídio é cometido mediante paga ou promessa de recompensa, ou por outro motivo torpe; 
        por motivo fútil; com emprego de veneno, fogo, explosivo, asfixia, tortura ou outro meio insidioso; 
        à traição, de emboscada; para assegurar a execução de outro crime; contra menor de 14 anos; 
        no contexto de violência doméstica (feminicídio): Pena - reclusão, de doze a trinta anos. 
        §7º A pena do feminicídio é aumentada de 1/3 até a metade.""",
        "data_publicacao": "2019-12-24",
        "tipo": "legislacao",
        "vigente": True
    },
    # --- Tráfico de Drogas ---
    {
        "id": "lei_6368_1976",
        "titulo": "Lei 6.368/1976 (Revogada) - Tráfico de Entorpecentes",
        "texto": """Art. 12: Importar ou exportar, remeter, preparar, produzir, fabricar, adquirir, 
        vender, expor à venda ou oferecer, fornecer ainda que gratuitamente, ter em depósito, transportar, 
        trazer consigo, guardar, prescrever, ministrar ou entregar, de qualquer forma, a consumo 
        substância entorpecente ou que determine dependência física ou psíquica, sem autorização 
        ou em desacordo com determinação legal ou regulamentar: Pena - Reclusão, de 3 a 15 anos, 
        e pagamento de 50 a 360 dias-multa.""",
        "data_publicacao": "1976-10-21",
        "tipo": "legislacao",
        "vigente": False,
        "revogado_por": "Lei 11.343/2006"
    },
    {
        "id": "lei_11343_2006",
        "titulo": "Lei 11.343/2006 - Lei de Drogas (Vigente)",
        "texto": """Art. 33: Importar, exportar, remeter, preparar, produzir, fabricar, adquirir, 
        vender, expor à venda, oferecer, ter em depósito, transportar, trazer consigo, guardar, 
        prescrever, ministrar, entregar a consumo ou fornecer drogas, ainda que gratuitamente, 
        sem autorização ou em desacordo com determinação legal ou regulamentar: Pena - reclusão 
        de 5 (cinco) a 15 (quinze) anos e pagamento de 500 (quinhentos) a 1.500 (mil e quinhentos) 
        dias-multa. §4º: pena reduzida de 1/6 a 2/3 para agente primário.""",
        "data_publicacao": "2006-08-23",
        "tipo": "legislacao",
        "vigente": True
    },
    # --- Jurisprudência ---
    {
        "id": "stf_reu_2015",
        "titulo": "STF - HC 126.292 - Execução Antecipada (2015, CANCELADA)",
        "texto": """CONSTITUCIONAL. HABEAS CORPUS. PRINCÍPIO CONSTITUCIONAL DA PRESUNÇÃO DE 
        INOCÊNCIA (CF, ART. 5º, LVII). ACÓRDÃO PENAL CONDENATÓRIO. EXECUÇÃO PROVISÓRIA. 
        POSSIBILIDADE. A execução provisória de acórdão penal condenatório proferido em grau 
        de apelação, ainda que sujeito a recurso especial ou extraordinário, não compromete 
        o princípio constitucional da presunção de inocência. Plenário, 17/02/2016.""",
        "data_publicacao": "2016-02-17",
        "tipo": "jurisprudencia",
        "vigente": False,
        "revogado_por": "ADC 43/44/54 STF 2019"
    },
    {
        "id": "stf_reu_2019",
        "titulo": "STF - ADC 43/44/54 - Presunção de Inocência (2019, VIGENTE)",
        "texto": """CONSTITUCIONAL. AÇÃO DECLARATÓRIA DE CONSTITUCIONALIDADE. ART. 283 DO CPP. 
        O Plenário do STF, por maioria, declarou constitucional o art. 283 do CPP, que condiciona 
        o início do cumprimento da pena ao trânsito em julgado da condenação criminal, afirmando 
        ser incompatível com a Constituição a chamada 'execução antecipada da pena'. 
        Julgado em 07/11/2019. Efeito vinculante.""",
        "data_publicacao": "2019-11-07",
        "tipo": "jurisprudencia",
        "vigente": True
    },
    {
        "id": "stj_trafico_2022",
        "titulo": "STJ - Tema 1051 - Tráfico e Tornozeleira (2022)",
        "texto": """RECURSO ESPECIAL REPETITIVO. TRÁFICO DE DROGAS. PRISÃO DOMICILIAR. 
        MONITORAÇÃO ELETRÔNICA. O STJ firmou o entendimento de que é possível, em caráter 
        excepcional, a substituição da prisão preventiva por prisão domiciliar com monitoração 
        eletrônica em crimes de tráfico de drogas, quando demonstrada situação excepcional 
        devidamente fundamentada. Tema 1051, julgado em 22/06/2022.""",
        "data_publicacao": "2022-06-22",
        "tipo": "jurisprudencia",
        "vigente": True
    },
    {
        "id": "cnj_resolucao_2024",
        "titulo": "CNJ - Resolução 525/2024 - Audiência de Custódia Digital",
        "texto": """Resolução CNJ n. 525, de 17 de janeiro de 2024. Dispõe sobre o procedimento 
        de audiência de custódia realizada por videoconferência. Os tribunais poderão realizar 
        audiências de custódia por videoconferência em municípios que não dispõem de estrutura 
        física adequada, desde que garantida a presença do defensor no local de recolhimento 
        do preso. Vigência: imediata.""",
        "data_publicacao": "2024-01-17",
        "tipo": "resolucao",
        "vigente": True
    },
]

print(f"📅 Corpus temporal: {len(documentos_temporais)} documentos")
print("\nDistribuição temporal:")
for doc in sorted(documentos_temporais, key=lambda x: x["data_publicacao"]):
    vigencia = "✅ Vigente" if doc.get("vigente") else "❌ Revogado"
    print(f"  {doc['data_publicacao']} | {vigencia} | {doc['titulo'][:50]}")

## Etapa 2 — Implementação da Função de Decay

In [ ]:
import math
from datetime import datetime

def decay_exponencial(
    data_documento: str,
    data_referencia: str = None,
    scale_dias: int = 365,
    offset_dias: int = 30
) -> float:
    """
    Calcula o decay exponencial temporal para um documento.
    
    Fórmula: decay = exp(-λ × max(0, age_days - offset))
    onde λ = ln(2) / scale_dias  (half-life = scale_dias)
    
    Args:
        data_documento: Data de publicação do documento (YYYY-MM-DD)
        data_referencia: Data de referência (padrão: hoje)
        scale_dias: Half-life em dias. scale=365 → doc de 1 ano = 50% do score original
        offset_dias: Período de graça sem penalização (docs recentes não são penalizados)
    
    Returns:
        float entre 0.0 e 1.0
    
    Equivalente OpenSearch:
    "exp": {
        "data_publicacao": {
            "origin": "now",
            "scale": "365d",
            "offset": "30d",
            "decay": 0.5
        }
    }
    """
    if data_referencia is None:
        data_ref = datetime.now()
    else:
        data_ref = datetime.strptime(data_referencia, "%Y-%m-%d")
    
    data_doc = datetime.strptime(data_documento, "%Y-%m-%d")
    age_days = (data_ref - data_doc).days
    
    # Período de graça
    age_penalizado = max(0, age_days - offset_dias)
    
    # Lambda (taxa de decay): ln(2) / half_life
    lam = math.log(2) / scale_dias
    
    return math.exp(-lam * age_penalizado)


# Demonstração da função de decay
DATA_HOJE = "2024-12-01"

print("=" * 60)
print("FUNÇÃO DE DECAY EXPONENCIAL (scale=365 dias, offset=30 dias)")
print("=" * 60)
print(f"{'Documento':<35} {'Data':>12} {'Idade (dias)':>13} {'Decay':>8}")
print("-" * 60)

for doc in sorted(documentos_temporais, key=lambda x: x["data_publicacao"], reverse=True):
    data = doc["data_publicacao"]
    data_dt = datetime.strptime(data, "%Y-%m-%d")
    ref_dt = datetime.strptime(DATA_HOJE, "%Y-%m-%d")
    idade = (ref_dt - data_dt).days
    decay_val = decay_exponencial(data, DATA_HOJE, scale_dias=365)
    titulo_curto = doc['titulo'][:34]
    print(f"{titulo_curto:<35} {data:>12} {idade:>13,} {decay_val:>8.4f}")

print("\n💡 Interpretação:")
print("  • Decay = 1.0: documento dentro do período de graça (30 dias)")
print("  • Decay = 0.5: documento com exatamente 1 half-life de idade (365 dias)")
print("  • Decay ≈ 0.0: documento muito antigo, quase sem peso")

## Etapa 3 — Motor de Busca Time-Aware Local

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("📥 Carregando BGE-M3 para embeddings...")
modelo = SentenceTransformer("BAAI/bge-m3")
print("✅ BGE-M3 pronto")

# Gerar embeddings do corpus temporal
for doc in documentos_temporais:
    embedding = modelo.encode(
        doc["titulo"] + " " + doc["texto"],
        normalize_embeddings=True
    )
    doc["embedding"] = embedding

print(f"✅ Embeddings gerados para {len(documentos_temporais)} documentos")


class BuscaTemporalRAG:
    """
    Motor de busca semântica com função de decay temporal.
    
    Simula o comportamento do OpenSearch function_score com decay exponencial.
    """
    
    def __init__(self, documentos: list[dict], data_referencia: str = None):
        self.documentos = documentos
        self.data_referencia = data_referencia or datetime.now().strftime("%Y-%m-%d")
        self.embeddings = np.array([d["embedding"] for d in documentos])
    
    def buscar(
        self,
        query: str,
        k: int = 5,
        scale_dias: int = 365,
        offset_dias: int = 30,
        peso_temporal: float = 0.3,
        apenas_vigentes: bool = False
    ) -> list[dict]:
        """
        Busca com score combinado: semantico × (1-peso_temporal) + temporal × peso_temporal
        
        Args:
            query: Texto da busca
            k: Número de resultados
            scale_dias: Half-life do decay em dias
            offset_dias: Período de graça
            peso_temporal: 0.0 = ignora tempo | 1.0 = só considera tempo
            apenas_vigentes: Filtrar apenas documentos vigentes
        """
        # Score semântico
        q_emb = modelo.encode(query, normalize_embeddings=True)
        scores_semanticos = self.embeddings @ q_emb
        
        resultados = []
        for i, doc in enumerate(self.documentos):
            # Filtro de vigência
            if apenas_vigentes and not doc.get("vigente", True):
                continue
            
            # Score temporal
            score_temporal = decay_exponencial(
                doc["data_publicacao"],
                self.data_referencia,
                scale_dias=scale_dias,
                offset_dias=offset_dias
            )
            
            # Score combinado
            score_sem = float(scores_semanticos[i])
            score_final = score_sem * (1 - peso_temporal) + score_temporal * peso_temporal
            
            resultados.append({
                **{k: v for k, v in doc.items() if k != 'embedding'},
                "_score_semantico": score_sem,
                "_score_temporal": score_temporal,
                "_score_final": score_final
            })
        
        # Ordenar pelo score final
        resultados.sort(key=lambda x: x["_score_final"], reverse=True)
        return resultados[:k]


# Inicializar motor de busca
motor = BuscaTemporalRAG(documentos_temporais, data_referencia=DATA_HOJE)
print("\n✅ Motor de busca time-aware pronto!")

## Etapa 4 — Comparativo: Com e Sem Decay Temporal

In [ ]:
def exibir_comparativo(query: str, k: int = 5):
    """Exibe comparativo lado a lado: sem vs com decay temporal."""
    
    print(f"\n{'='*70}")
    print(f"🔍 QUERY: \"{query}\"")
    print(f"{'='*70}")
    
    # Sem decay (puro semântico)
    sem_decay = motor.buscar(query, k=k, peso_temporal=0.0)
    
    # Com decay (semântico + temporal)
    com_decay = motor.buscar(query, k=k, peso_temporal=0.3, scale_dias=365)
    
    # Apenas vigentes
    apenas_vigentes = motor.buscar(query, k=k, peso_temporal=0.3, apenas_vigentes=True)
    
    print(f"\n{'Pos':<4} {'SEM Decay (só semântico)':<28} {'COM Decay (temporal)':<28} {'APENAS VIGENTES':<28}")
    print("-" * 92)
    
    max_len = max(len(sem_decay), len(com_decay), len(apenas_vigentes))
    for pos in range(min(k, max_len)):
        def formatar_doc(resultados, idx):
            if idx >= len(resultados):
                return "-" * 27
            doc = resultados[idx]
            vigencia = "✅" if doc.get("vigente") else "❌"
            data = doc["data_publicacao"][:4]
            score = doc.get("_score_final", doc.get("_score_semantico", 0))
            titulo = doc['titulo'][:18]
            return f"{vigencia}{data} [{score:.3f}] {titulo}"
        
        col1 = formatar_doc(sem_decay, pos)
        col2 = formatar_doc(com_decay, pos)
        col3 = formatar_doc(apenas_vigentes, pos)
        print(f"{pos+1:<4} {col1:<28} {col2:<28} {col3:<28}")


# Demonstração com queries críticas
exibir_comparativo("tráfico de drogas pena reclusão")
exibir_comparativo("execução da pena presunção inocência")
exibir_comparativo("audiência de custódia preso juiz")

## Etapa 5 — Configuração OpenSearch (Código de Produção)

In [ ]:
# Código de produção para OpenSearch com time-aware RAG
# ATENÇÃO: Não executar no Colab sem servidor OpenSearch configurado

CONFIGURACAO_OPENSEARCH = '''
# ============================================================
# 1. CRIAR ÍNDICE COM CAMPO DE DATA
# ============================================================
PUT /juridico_temporal
{
  "settings": {
    "index": {"knn": true}
  },
  "mappings": {
    "properties": {
      "embedding": {
        "type": "knn_vector",
        "dimension": 1024,
        "method": {"name": "hnsw", "space_type": "cosinesimil"}
      },
      "data_publicacao": {"type": "date", "format": "yyyy-MM-dd"},
      "texto": {"type": "text", "analyzer": "portuguese"},
      "vigente": {"type": "boolean"},
      "tipo": {"type": "keyword"}
    }
  }
}

# ============================================================
# 2. BUSCA KNN COM DECAY EXPONENCIAL
# ============================================================
POST /juridico_temporal/_search
{
  "query": {
    "function_score": {
      "query": {
        "knn": {
          "embedding": {
            "vector": [0.12, -0.45, 0.78, ...],
            "k": 20
          }
        }
      },
      "functions": [
        {
          "exp": {
            "data_publicacao": {
              "origin": "now",
              "scale": "365d",
              "offset": "30d",
              "decay": 0.5
            }
          }
        },
        {
          "filter": {"term": {"vigente": true}},
          "weight": 2.0
        }
      ],
      "score_mode": "multiply",
      "boost_mode": "multiply"
    }
  },
  "size": 5
}

# ============================================================
# 3. VARIANTE: HARD FILTER APENAS VIGENTES
# ============================================================
POST /juridico_temporal/_search
{
  "query": {
    "bool": {
      "must": {
        "knn": {"embedding": {"vector": [...], "k": 10}}
      },
      "filter": [
        {"term": {"vigente": true}},
        {"range": {"data_publicacao": {"gte": "now-5y"}}}
      ]
    }
  }
}
'''

print("📋 Configuração OpenSearch Time-Aware RAG:")
print(CONFIGURACAO_OPENSEARCH)

## Etapa 6 — Calibração do Parâmetro Scale

In [ ]:
# Análise do impacto do parâmetro scale em diferentes domínios jurídicos

print("=" * 70)
print("CALIBRAÇÃO DO PARÂMETRO SCALE POR DOMÍNIO JURÍDICO")
print("=" * 70)

idades_teste = [30, 90, 180, 365, 730, 1825, 3650]
configuracoes = [
    {"nome": "Regulações Operacionais", "scale": 180, "offset": 7},
    {"nome": "Legislação Geral",         "scale": 365, "offset": 30},
    {"nome": "Jurisprudência Recente",    "scale": 730, "offset": 60},
    {"nome": "Doutrina Jurídica",         "scale": 1825, "offset": 90},
]

print(f"\n{'Idade (dias)':>12}", end="")
for cfg in configuracoes:
    print(f" | {cfg['nome']:<23}", end="")
print()
print("-" * 105)

for idade in idades_teste:
    data_doc = (datetime.strptime(DATA_HOJE, "%Y-%m-%d") - timedelta(days=idade)).strftime("%Y-%m-%d")
    print(f"{idade:>12}", end="")
    for cfg in configuracoes:
        d = decay_exponencial(data_doc, DATA_HOJE, cfg["scale"], cfg["offset"])
        print(f" | {d:>23.4f}", end="")
    print()

print("""
Recomendações por domínio:
  • Portarias e atos administrativos: scale=180d (perdem relevância rápido)
  • Legislação: scale=365d (mudanças frequentes mas não tão rápidas)
  • Súmulas STJ/STF: scale=730d (longa vigência, mudanças raras)
  • Doutrina e artigos: scale=1825d (referência mesmo sendo antiga)
  
  Estratégia híbrida recomendada:
  1. Hard filter: remover documentos explicitamente revogados (campo vigente=false)
  2. Soft decay: penalizar documentos antigos sem revogar explicitamente
  3. Boost: adicionar peso extra para documentos marcados como 'vigente=true'
""")

## Exercício Prático

> **Exercício:** Adicione 3 documentos ao corpus: uma portaria de 2023, uma resolução de 2024 e uma resolução revogada de 2021. Execute a query *"procedimento audiência de custódia videoconferência"* com diferentes valores de `scale_dias` (90, 365, 730) e observe como os resultados mudam. Documente qual configuração seria mais adequada para um sistema de assessoria jurídica em tempo real.

## Pergunta de Reflexão

> **Reflexão:** Um sistema Time-Aware com decay muito agressivo pode ignorar princípios jurídicos fundamentais que não mudam com o tempo (como o princípio da dignidade humana ou presunção de inocência). Como você projetaria um sistema que prioriza recência para regulamentos operacionais mas mantém documentos de direitos fundamentais sempre com score máximo?

---

**Próximo:** Lab 5 — DSPy: Otimização Automática de Pipeline RAG